<!-- SPDX-FileCopyrightText: Copyright (c) 2026 Super Protocol. All rights reserved.
     SPDX-License-Identifier: Apache-2.0 -->

# 00: What this deployment actually has

This notebook is ours, not a port. Run it first, every time — it is the only cell sequence in
this workspace whose job is to **fail loudly** rather than to produce a result.

Four facts, in the order that a wrong one costs the most:

1. **A GPU is visible.** The tokenizer is cuDF and the model is torch; without a card nothing
   here runs, and the failure forty cells into notebook 01 looks like a bug in the notebook.
2. **The checkpoint in the image loads, and the tokenizer agrees with it.** The shipped model's
   embedding table has one row per token of a 6,251-token vocabulary. A tokenizer that built a
   different vocabulary would produce ids that index someone else's embeddings — no error, just
   numbers that mean nothing.
3. **Your columns are where this deployment was told they are.** The mapping comes from the
   deployment's parameters; a column named here that your data does not have is a message in
   this notebook rather than a `KeyError` in the middle of a corpus build.
4. **Your data lands inside that vocabulary.** This is the one that has no counterpart
   upstream and the one worth reading. The shipped checkpoint was pretrained on a public US
   card-transaction corpus: US states, 3-digit ZIP prefixes, a fixed list of merchant category
   codes, dollar-amount buckets and customer indices 0–2999. Transactions from outside that
   world tokenise entirely to catch-alls — `STATE_XX`, `MCC_-1`, `CAT_GENERAL` — and every
   notebook downstream still produces embeddings, a UMAP and an AUROC from them.

If the last check fails, nothing is broken and you have not wasted the deployment: notebook 03
pretrains with a vocabulary built from **your** fields and does not use the shipped weights at
all. What you cannot sensibly do is extract embeddings from a tokenizer that matched nothing.

In [ ]:
# 1. The card.
import tfm_workspace as tfm

print("GPU:        ", tfm.require_gpu())

import torch
print("torch:      ", torch.__version__)
print("devices:    ", torch.cuda.device_count())

In [ ]:
# 2. The checkpoint in the image, and whether the tokenizer agrees with it.
#
# Loaded from its own bytes rather than from its config.json: the fork's image
# build verifies the sha256 of the safetensors file, and this is the same file
# being asked to become a model.
import json
from pathlib import Path

from transformers import AutoModelForCausalLM

from src.tokenizer import FinancialTabularTokenizer

print(tfm.describe_checkpoint())

config = json.loads((Path(tfm.MODEL_DIR) / "config.json").read_text())
model = AutoModelForCausalLM.from_pretrained(tfm.MODEL_DIR, dtype="float32")
parameters = sum(parameter.numel() for parameter in model.parameters())
rows = model.get_input_embeddings().weight.shape[0]

tokenizer = FinancialTabularTokenizer()

print(f"parameters:            {parameters:,}")
print(f"embedding table rows:  {rows:,}")
print(f"tokenizer vocabulary:  {tokenizer.vocab_size:,}")

assert tokenizer.vocab_size == rows == config["vocab_size"], (
    f"the tokenizer builds {tokenizer.vocab_size} tokens and this checkpoint was trained with "
    f"{rows}: every id this tokenizer produces would index the wrong row"
)
print("\nthe tokenizer and the checkpoint agree on the vocabulary")

# The model object itself is not needed below; 04 loads it properly, on the card.
del model

In [ ]:
# 3. Where the data comes from, and what this deployment was told the columns are.
print(tfm.describe_binding())
print()
print(tfm.describe_columns())
print()
print(f"row ceiling (TFM_MAX_ROWS):  {tfm.MAX_ROWS:,}")
print(f"coverage floor:              {tfm.MIN_COVERAGE:.0%}")

In [ ]:
# 4. The data itself, and how much of it the shipped vocabulary has ever seen.
#
# A sample, not the whole history: this is a diagnostic, and the point is to find
# out in a minute rather than after an hour of tokenising. The notebooks that do
# the work read everything.
SAMPLE_ROWS = 500_000

frame = tfm.load_transactions(limit=SAMPLE_ROWS)
print(f"{len(frame):,} rows x {frame.shape[1]} columns")
print()
frame.head(5).to_pandas()

In [ ]:
# This is the cell that either passes or tells you exactly what is wrong.
#
# It raises below the deployment's floor. tfm.check_coverage(frame, minimum=0)
# overrides it deliberately — and if you do that, nothing downstream means what
# it appears to mean.
report = tfm.check_coverage(frame)

print()
print("ready. Run 01_dataset_baseline.ipynb next" if report["labelled"] else
      "ready. No fraud label, so start at 02_seq_preproc_tokenization.ipynb: "
      "01 and 05 score fraud detection and have nothing to score.")

## If the coverage check failed

Read the per-field numbers it printed; they say which part of the vocabulary your data missed.

| What missed | What it means | What to do |
|---|---|---|
| `state` | the field is not a US state abbreviation | If you have a region column that is one, name it in the deployment's **State column**. Otherwise this field tokenises to `STATE_XX` for every row — one token, no information. |
| `mcc` | your category codes are not in the corpus's fixed MCC list | Map your scheme onto ISO 18245 MCCs in a cell of your own, write the result to `/workspace/derived`, and read that. Or accept `MCC_-1` and pretrain in 03. |
| `zip3` | the postcode has no 3-digit numeric prefix (any non-US scheme) | Nothing to map; pretrain in 03 instead. |
| `user` | customer ids outside 0–2999 | The vocabulary has 3,000 customer tokens, and anything above 2,999 is **clipped onto the same token**, so unrelated customers share an identity. Re-index your customers 0..N-1 if N is under 3,000; otherwise pretrain in 03, where the vocabulary is built from your data. |
| `card` | more than ten cards per customer | Same shape of problem, smaller: ids above 9 collapse onto `CARD_9`. |
| `entrymode` | your entry mode is not `SWIPE`/`CHIP`/`ONLINE TRANSACTION` | Map your values onto those three strings in the deployment's **Entry-mode column**, or accept `CHIP_UNK`. |

**Pretraining in 03 is not the fallback, it is the other half of the product.** The shipped
checkpoint is NVIDIA's, trained on a public corpus; a model trained in here is trained on your
transactions, inside the enclave, and the deployment publishes signed evidence of exactly which
code touched them. What the shipped weights are for is a starting point when your data does fit
their world, and a baseline to beat when it does not.